# 02. Data Cleansing & Feature Engineering

This notebook implements the data sanitization pipeline established during the exploratory audit.

---
### Pipeline Stages:
1. Standardize schema column headers and identifier conventions.
2. Filter out transactions lacking a valid customer identifier (guest checkouts).
3. Exclude order cancellations and credit reversals (invoices prefixed with 'C').
4. Filter out negative and zero quantities (damaged inventory and customer returns).
5. Filter out negative and zero unit prices (system test records and manual adjustments).
6. De-duplicate transaction rows and cast attributes into strict analytical types.
7. Engineer line-item gross revenue (`TotalRevenue = Quantity * Price`).
8. Export the sanitized fact table to `data/retail_cleaned.csv`.


### Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')


### Step 2: Configure Display Settings

In [2]:
pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', lambda x: '%.2f' % x)


### Step 3: Load Fiscal Period 1 (Year 2009–2010)

In [3]:
data_path = 'data/online_retail_II.xlsx'
df_2009 = pd.read_excel(data_path, sheet_name='Year 2009-2010', engine='openpyxl')
print(f'Year 2009-2010 loaded: {len(df_2009):,} rows')


Year 2009-2010 loaded: 525,461 rows


### Step 4: Load Fiscal Period 2 (Year 2010–2011)

In [4]:
df_2010 = pd.read_excel(data_path, sheet_name='Year 2010-2011', engine='openpyxl')
print(f'Year 2010-2011 loaded: {len(df_2010):,} rows')


Year 2010-2011 loaded: 541,910 rows


### Step 5: Merge Partitions into Master Raw DataFrame

In [5]:
df = pd.concat([df_2009, df_2010], ignore_index=True)
raw_row_count = len(df)
print(f'Master raw dataset: {raw_row_count:,} records')


Master raw dataset: 1,067,371 records


### Step 6: Inspect Raw Schema Column Names

In [6]:
list(df.columns)


['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'Customer ID',
 'Country']

### Step 7: Standardize Column Names
Rename 'Customer ID' to 'CustomerID' and trim any whitespace.

In [7]:
df.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)
df.columns = df.columns.str.strip()
print('Standardized column names:', list(df.columns))


Standardized column names: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country']


### Step 8: Quantify Missing Customer Identifiers

In [8]:
missing_cust_count = df['CustomerID'].isnull().sum()
print(f'Rows missing CustomerID: {missing_cust_count:,} ({(missing_cust_count * 100 / len(df)):.2f}%)')


Rows missing CustomerID: 243,007 (22.77%)


### Step 9: Filter Out Transactions Lacking Customer Identifiers

In [9]:
df = df[df['CustomerID'].notnull()].copy()
print(f'Remaining records with valid CustomerID: {len(df):,}')


Remaining records with valid CustomerID: 824,364


### Step 10: Cast CustomerID to Clean Identifier String

In [10]:
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
print('Sample CustomerIDs:', df['CustomerID'].head(3).tolist())


Sample CustomerIDs: ['13085', '13085', '13085']


### Step 11: Identify Cancellation Invoices (Prefixed with 'C')

In [11]:
cancellations = df[df['Invoice'].astype(str).str.startswith('C')]
print(f'Cancellation transactions detected: {len(cancellations):,}')


Cancellation transactions detected: 18,744


### Step 12: Exclude Cancellation Invoices

In [12]:
df = df[~df['Invoice'].astype(str).str.startswith('C')].copy()
print(f'Records remaining after removing cancellations: {len(df):,}')


Records remaining after removing cancellations: 805,620


### Step 13: Detect Non-Positive Quantities (<= 0)

In [13]:
non_pos_qty = df[df['Quantity'] <= 0]
print(f'Non-positive quantity rows found: {len(non_pos_qty):,}')


Non-positive quantity rows found: 0


### Step 14: Filter for Strictly Positive Quantities

In [14]:
df = df[df['Quantity'] > 0].copy()
print(f'Records remaining with Quantity > 0: {len(df):,}')


Records remaining with Quantity > 0: 805,620


### Step 15: Detect Non-Positive Unit Prices (<= 0)

In [15]:
non_pos_price = df[df['Price'] <= 0]
print(f'Non-positive price rows found: {len(non_pos_price):,}')


Non-positive price rows found: 71


### Step 16: Filter for Strictly Positive Unit Prices

In [16]:
df = df[df['Price'] > 0].copy()
print(f'Records remaining with Price > 0: {len(df):,}')


Records remaining with Price > 0: 805,549


### Step 17: Identify and Remove Exact Duplicate Transactions

In [17]:
duplicates_found = df.duplicated().sum()
print(f'Duplicate transaction rows detected: {duplicates_found:,}')
df.drop_duplicates(inplace=True)
print(f'Records remaining after deduplication: {len(df):,}')


Duplicate transaction rows detected: 26,124


Records remaining after deduplication: 779,425


### Step 18: Parse and Format Transaction Timestamps

In [18]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Verified date range: {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')


Verified date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


### Step 19: Extract Monthly Cohort Period (InvoiceMonth)

In [19]:
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')
df[['InvoiceDate', 'InvoiceMonth']].head(3)


,InvoiceDate,InvoiceMonth
0,2009-12-01 07:45:00,2009-12
1,2009-12-01 07:45:00,2009-12
2,2009-12-01 07:45:00,2009-12


### Step 20: Calculate Line-Item Gross Revenue (`TotalRevenue = Quantity * Price`)

In [20]:
df['TotalRevenue'] = (df['Quantity'] * df['Price']).round(2)
df[['Quantity', 'Price', 'TotalRevenue']].head()


,Quantity,Price,TotalRevenue
0,12,6.95,83.40
1,12,6.75,81.00
2,12,6.75,81.00
3,48,2.10,100.80
4,24,1.25,30.00


### Step 21: Clean and Normalize Product Descriptions

In [21]:
df['Description'] = df['Description'].astype(str).str.strip().str.upper()
df[['StockCode', 'Description']].head(3)


,StockCode,Description
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,79323P,PINK CHERRY LIGHTS
2,79323W,WHITE CHERRY LIGHTS


### Step 22: Final Data Quality Inspection and Summary

In [22]:
print('Final Null Count by Column:')
print(df.isnull().sum())
print(f'\nTotal Retained Valid Records: {len(df):,} of {raw_row_count:,} ({(len(df)*100/raw_row_count):.2f}% retention)')


Final Null Count by Column:


Invoice         0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
Price           0
CustomerID      0
Country         0
InvoiceMonth    0
TotalRevenue    0
dtype: int64

Total Retained Valid Records: 779,425 of 1,067,371 (73.02% retention)


### Step 23: Summary Statistics of Clean Financial Metrics

In [23]:
summary_table = pd.Series({
    'Total Completed Invoices': df['Invoice'].nunique(),
    'Total Unique Customers' : df['CustomerID'].nunique(),
    'Total Unique SKUs'      : df['StockCode'].nunique(),
    'Total Gross Revenue ($)': f'${df["TotalRevenue"].sum():,.2f}',
    'Average Line Value ($)' : f'${df["TotalRevenue"].mean():.2f}'
})
summary_table


Total Completed Invoices             36969
Total Unique Customers                5878
Total Unique SKUs                     4631
Total Gross Revenue ($)     $17,374,804.25
Average Line Value ($)              $22.29
dtype: object

### Step 24: Export Clean Fact Table to CSV

In [24]:
output_path = 'data/retail_cleaned.csv'
df.to_csv(output_path, index=False)
print(f'Successfully exported sanitized fact table to {output_path} ({len(df):,} rows).')


Successfully exported sanitized fact table to data/retail_cleaned.csv (779,425 rows).
